# Automatic Circle Coordinate Extraction from Engineering Drawings

Computer-vision pipeline that:
1. Detects the outer square boundary and crops the ROI (**FR-1**)
2. Detects all circular markers (**FR-2**)
3. Detects measurement lines / arrows (**FR-3**)
4. Flags circles that already have a line passing through them (**FR-4**)
5. Picks a reference circle → origin `(0,0)` (**FR-5**)
6. Computes relative pixel coordinates (**FR-6**)
7. Converts pixels → millimetres via calibration (**FR-7**)
8. Exports `coordinates.csv` (**FR-8**)
9. Saves an annotated `result.png` (**FR-9**)

The classical circle-detection step (`detect_circles`) is isolated so it can later be swapped for a deep-learning detector (YOLO / RF-DETR / keypoints) without touching the rest of the pipeline.

## 0. Setup

Run once if the packages are missing.

In [ ]:
# !pip install opencv-python numpy pandas matplotlib

import os
import glob
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('OpenCV', cv2.__version__)

## 1. Configuration (Section 5 — Configurable Parameters)

All tunable knobs live in one place. Adjust these to match your drawing scale/resolution.

In [ ]:
@dataclass
class Config:
    # --- I/O ---
    input_folder: str = 'input'
    output_folder: str = 'output'

    # --- FR-2: Circle detection (Hough) ---
    min_radius: int = 6          # Minimum Circle Radius (px)
    max_radius: int = 40         # Maximum Circle Radius (px)
    min_center_dist: int = 20    # min distance between two circle centers (px)
    hough_param1: int = 100      # Canny high threshold inside HoughCircles
    hough_param2: int = 25       # accumulator threshold (lower => more circles)

    # --- FR-3: Line / arrow detection (Hough lines) ---
    min_line_length: int = 40    # Minimum Line Length (px)
    max_line_gap: int = 10       # max gap to link collinear segments (px)
    line_thickness_thresh: int = 60  # Canny threshold for line edges

    # --- FR-4: Proximity test ---
    line_tolerance: int = 8      # Line Proximity Tolerance (px)

    # --- FR-1: ROI detection ---
    roi_min_area_frac: float = 0.10  # square must cover >=10% of image
    use_roi: bool = True             # set False to skip cropping

    # --- FR-5: Reference circle strategy ---
    #   'nearest_center' | 'top_left' | 'largest'
    reference_strategy: str = 'nearest_center'

    # --- FR-7: Pixel -> mm calibration ---
    calib_pixels: float = 845.0
    calib_mm: float = 11.0

    @property
    def mm_per_pixel(self) -> float:
        return self.calib_mm / self.calib_pixels

CFG = Config()
print(CFG)
print('mm_per_pixel =', round(CFG.mm_per_pixel, 6))

## Data model

In [ ]:
@dataclass
class Circle:
    id: int
    x: int          # center x in ROI coordinates
    y: int          # center y in ROI coordinates
    r: int          # radius
    measured: bool = False
    is_reference: bool = False
    dx: float = 0.0     # relative px (FR-6)
    dy: float = 0.0
    x_mm: float = 0.0   # relative mm (FR-7)
    y_mm: float = 0.0

Line = Tuple[int, int, int, int]  # x1, y1, x2, y2

## FR-1 — Detect square boundary & crop ROI

In [ ]:
def detect_roi(gray: np.ndarray, cfg: Config) -> Tuple[np.ndarray, Tuple[int, int]]:
    """Find the largest 4-sided contour (the drawing square) and crop it.

    Returns the cropped grayscale ROI and the (x_offset, y_offset) of the crop
    within the original image so results can be mapped back if needed.
    """
    h, w = gray.shape
    if not cfg.use_roi:
        return gray, (0, 0)

    # Binarize: black lines -> white foreground on black background
    _, thresh = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    best = None
    best_area = cfg.roi_min_area_frac * h * w
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        area = cv2.contourArea(c)
        if len(approx) == 4 and area > best_area:
            best_area = area
            best = approx

    if best is None:
        # No clear square found -> use whole image
        return gray, (0, 0)

    x, y, bw, bh = cv2.boundingRect(best)
    # Shrink slightly inward so the border line itself is excluded
    pad = 3
    x0, y0 = x + pad, y + pad
    x1, y1 = x + bw - pad, y + bh - pad
    roi = gray[y0:y1, x0:x1]
    return roi, (x0, y0)

## FR-2 — Detect all circles

> **Swap point for a DL model.** Replace the body of `detect_circles` with a YOLO / RF-DETR / keypoint inference that returns the same `List[Circle]`; nothing downstream changes.

In [ ]:
def detect_circles(gray_roi: np.ndarray, cfg: Config) -> List[Circle]:
    """Classical Hough-circle detection. Returns circles in ROI coordinates."""
    blur = cv2.medianBlur(gray_roi, 3)
    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=cfg.min_center_dist,
        param1=cfg.hough_param1,
        param2=cfg.hough_param2,
        minRadius=cfg.min_radius,
        maxRadius=cfg.max_radius,
    )

    result: List[Circle] = []
    if circles is not None:
        circles = np.round(circles[0]).astype(int)
        for i, (x, y, r) in enumerate(circles, start=1):
            result.append(Circle(id=i, x=int(x), y=int(y), r=int(r)))
    return result

## FR-3 — Detect measurement lines / arrows

In [ ]:
def detect_lines(gray_roi: np.ndarray, cfg: Config) -> List[Line]:
    """Detect straight measurement lines (horizontal, vertical, diagonal)."""
    edges = cv2.Canny(gray_roi, cfg.line_thickness_thresh,
                      cfg.line_thickness_thresh * 2)
    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=50,
        minLineLength=cfg.min_line_length,
        maxLineGap=cfg.max_line_gap,
    )
    out: List[Line] = []
    if lines is not None:
        for l in lines[:, 0, :]:
            out.append((int(l[0]), int(l[1]), int(l[2]), int(l[3])))
    return out

## FR-4 — Determine measured circles

A circle is *measured* if any line's closest point to the circle centre is within `radius + tolerance`.

In [ ]:
def _point_to_segment_dist(px: float, py: float, line: Line) -> float:
    """Shortest distance from point (px,py) to segment (x1,y1)-(x2,y2)."""
    x1, y1, x2, y2 = line
    dx, dy = x2 - x1, y2 - y1
    seg_len_sq = dx * dx + dy * dy
    if seg_len_sq == 0:
        return float(np.hypot(px - x1, py - y1))
    t = ((px - x1) * dx + (py - y1) * dy) / seg_len_sq
    t = max(0.0, min(1.0, t))
    proj_x = x1 + t * dx
    proj_y = y1 + t * dy
    return float(np.hypot(px - proj_x, py - proj_y))


def flag_measured(circles: List[Circle], lines: List[Line], cfg: Config) -> None:
    """Set circle.measured = True in place when a line passes through/near it."""
    for c in circles:
        for line in lines:
            d = _point_to_segment_dist(c.x, c.y, line)
            if d < c.r + cfg.line_tolerance:
                c.measured = True
                break

## FR-5 — Detect reference circle (origin)

In [ ]:
def select_reference(circles: List[Circle], roi_shape: Tuple[int, int],
                     cfg: Config) -> Optional[Circle]:
    """Choose the reference circle among *unmeasured* circles."""
    candidates = [c for c in circles if not c.measured] or circles
    if not candidates:
        return None

    h, w = roi_shape
    if cfg.reference_strategy == 'nearest_center':
        cx, cy = w / 2.0, h / 2.0
        ref = min(candidates, key=lambda c: (c.x - cx) ** 2 + (c.y - cy) ** 2)
    elif cfg.reference_strategy == 'top_left':
        ref = min(candidates, key=lambda c: c.x + c.y)
    elif cfg.reference_strategy == 'largest':
        ref = max(candidates, key=lambda c: c.r)
    else:
        raise ValueError(f'Unknown reference_strategy: {cfg.reference_strategy}')

    ref.is_reference = True
    return ref

## FR-6 & FR-7 — Relative coordinates and mm conversion

In [ ]:
def compute_coordinates(circles: List[Circle], ref: Circle, cfg: Config) -> None:
    """Fill dx/dy (px) and x_mm/y_mm for every circle, relative to ref.

    Image y grows downward, so dy is flipped to a conventional upward axis.
    """
    scale = cfg.mm_per_pixel
    for c in circles:
        c.dx = float(c.x - ref.x)
        c.dy = float(ref.y - c.y)   # flip so up is positive
        c.x_mm = round(c.dx * scale, 3)
        c.y_mm = round(c.dy * scale, 3)

## FR-8 — CSV export

In [ ]:
def export_csv(circles: List[Circle], path: str) -> pd.DataFrame:
    rows = []
    for c in circles:
        rows.append({
            'Circle': c.id,
            'X(px)': int(c.dx),
            'Y(px)': int(c.dy),
            'X(mm)': c.x_mm,
            'Y(mm)': c.y_mm,
            'Measured': 'Yes' if c.measured else 'No',
            'Reference': 'Yes' if c.is_reference else 'No',
        })
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df

## FR-9 — Visualization

Green = detected circles · Blue = reference · Red = measured · Yellow = lines

In [ ]:
def visualize(gray_roi: np.ndarray, circles: List[Circle], lines: List[Line],
              path: str) -> np.ndarray:
    canvas = cv2.cvtColor(gray_roi, cv2.COLOR_GRAY2BGR)

    # Lines (yellow)
    for x1, y1, x2, y2 in lines:
        cv2.line(canvas, (x1, y1), (x2, y2), (0, 255, 255), 1)

    for c in circles:
        if c.is_reference:
            color = (255, 0, 0)      # blue
        elif c.measured:
            color = (0, 0, 255)      # red
        else:
            color = (0, 200, 0)      # green
        cv2.circle(canvas, (c.x, c.y), c.r, color, 2)
        cv2.circle(canvas, (c.x, c.y), 2, color, -1)
        cv2.putText(canvas, str(c.id), (c.x + c.r + 2, c.y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

    cv2.imwrite(path, canvas)
    return canvas

## 7. Full processing pipeline

In [ ]:
def process_image(image_path: str, cfg: Config, show: bool = True):
    """Run the full pipeline on one image. Returns (DataFrame, annotated BGR)."""
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # FR-1
    roi, _offset = detect_roi(gray, cfg)
    # FR-2
    circles = detect_circles(roi, cfg)
    # FR-3
    lines = detect_lines(roi, cfg)
    # FR-4
    flag_measured(circles, lines, cfg)
    # FR-5
    ref = select_reference(circles, roi.shape, cfg)
    if ref is None:
        print(f'[warn] no circles detected in {image_path}')
        return pd.DataFrame(), roi
    # FR-6 + FR-7
    compute_coordinates(circles, ref, cfg)

    # FR-8 + FR-9
    os.makedirs(cfg.output_folder, exist_ok=True)
    stem = os.path.splitext(os.path.basename(image_path))[0]
    csv_path = os.path.join(cfg.output_folder, f'{stem}_coordinates.csv')
    png_path = os.path.join(cfg.output_folder, f'{stem}_result.png')

    df = export_csv(circles, csv_path)
    canvas = visualize(roi, circles, lines, png_path)

    print(f'{image_path}: {len(circles)} circles, '
          f'{sum(c.measured for c in circles)} measured, '
          f'{len(lines)} lines -> {csv_path}')

    if show:
        plt.figure(figsize=(9, 9))
        plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        plt.title(f'{stem} — green:unmeasured  blue:ref  red:measured  yellow:lines')
        plt.axis('off')
        plt.show()

    return df, canvas


def process_folder(cfg: Config):
    """Batch over every image in the input folder (Non-functional: scalability)."""
    patterns = ('*.png', '*.jpg', '*.jpeg')
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(cfg.input_folder, p)))
    files.sort()
    if not files:
        print(f'No images found in {cfg.input_folder}/')
    for f in files:
        process_image(f, cfg, show=False)

## 8. Run on a single image

Point this at one of your drawings and tune the `Config` values above until the overlay looks right.

In [ ]:
# df, canvas = process_image('input/image1.png', CFG)
# df

## 9. Synthetic demo (no input image required)

Generates a test drawing so you can verify the whole pipeline end-to-end before using real CAD exports.

In [ ]:
def make_synthetic(path='input/demo.png', size=600):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    img = np.full((size, size), 255, np.uint8)

    # Outer square boundary
    cv2.rectangle(img, (40, 40), (size - 40, size - 40), 0, 2)

    centers = [(150, 150), (300, 150), (450, 160),
               (180, 300), (320, 320), (470, 300),
               (200, 460), (360, 470)]
    for (x, y) in centers:
        cv2.circle(img, (x, y), 14, 0, 2)

    # A horizontal measurement arrow that passes through two circles (150,150)-(300,150)
    cv2.arrowedLine(img, (150, 150), (300, 150), 0, 2, tipLength=0.05)
    # A vertical arrow through (180,300)-(200,460)-ish
    cv2.arrowedLine(img, (180, 300), (200, 460), 0, 2, tipLength=0.05)

    cv2.imwrite(path, img)
    return path

demo_path = make_synthetic()
df, _ = process_image(demo_path, CFG)
df

## 10. Batch mode (Section 6 folder structure)

Drop all drawings into `input/` and run:

In [ ]:
# process_folder(CFG)

---
### Tuning guide

| Symptom | Fix |
|---|---|
| Missing circles | lower `hough_param2`, widen `min_radius`/`max_radius` |
| False circles | raise `hough_param2`, raise `min_center_dist` |
| Circles wrongly flagged *measured* | lower `line_tolerance` |
| Measured circles not flagged | raise `line_tolerance`, lower `min_line_length` |
| ROI crop wrong | set `use_roi=False` or adjust `roi_min_area_frac` |
| Wrong mm values | recalibrate `calib_pixels` / `calib_mm` from a known dimension |

**DL upgrade path:** replace only `detect_circles()` with a model that returns `List[Circle]` (id, x, y, r in ROI coords). FR-3 … FR-9 stay untouched.